In [ ]:
# ================================
# Match sale & return at item level
# ================================
import pandas as pd
import json
import re
from google.colab import files

# ================================
# UPLOAD FILE
# ================================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

if file_name.endswith('.csv'):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name)

print("File Loaded:", df.shape)

# ================================
# HELPER FUNCTIONS
# ================================
def clean_phone(val):
    if pd.isna(val):
        return None
    val = str(val)
    val = re.sub(r'\.0$', '', val)
    val = re.sub(r'\D', '', val)
    return val[-10:] if len(val) >= 10 else None

def extract_email(val):
    if pd.isna(val):
        return None
    val = str(val)
    return val.lower() if "@" in val else None

def safe_json(x):
    try:
        return json.loads(x)
    except:
        return []

# ================================
# CLASSIFY SALE / RETURN
# ================================
def classify(row):
    t = str(row.get("event_props.Invoice Type", "")).lower()

    if "return" in t or "refund" in t:
        return "return"

    try:
        val = row.get("event_props.Total Billing Value", "")
        if float(str(val).replace(",", "")) < 0:
            return "return"
    except:
        pass

    return "sale"

df["inv_type_norm"] = df.apply(classify, axis=1)

# ================================
# EXPLODE ITEMS
# ================================
df['items_parsed'] = df['items'].apply(safe_json)
df = df.explode('items_parsed').reset_index(drop=True)

# ================================
# ITEM EXTRACTION
# ================================
df['Invoice Brand'] = df['items_parsed'].apply(lambda x: x.get('Items|Invoice Brand') if isinstance(x, dict) else None)
df['Model Number'] = df['items_parsed'].apply(lambda x: x.get('Items|Model Number') if isinstance(x, dict) else None)
df['MRP'] = df['items_parsed'].apply(lambda x: x.get('Items|MRP') if isinstance(x, dict) else None)
df['Billing Value'] = df['items_parsed'].apply(lambda x: x.get('Items|Billing Value') if isinstance(x, dict) else None)
df['Billing Discount Percentage'] = df['items_parsed'].apply(lambda x: x.get('Items|Billing Discount Percentage') if isinstance(x, dict) else None)

# ================================
# CREATE MATCH KEY (IMPORTANT)
# ================================
df['match_key'] = (
    df['event_props.CUID Number'].astype(str) + "||" +
    df['Invoice Brand'].astype(str) + "||" +
    df['Model Number'].astype(str)
)

# ================================
# SALE vs RETURN MATCHING
# ================================
df['Billing Value'] = pd.to_numeric(df['Billing Value'], errors='coerce').abs()

to_remove = []

for key, grp in df.groupby('match_key'):

    sales = grp[grp['inv_type_norm'] == 'sale'].sort_values('ts')
    returns = grp[grp['inv_type_norm'] == 'return'].sort_values('ts')

    n = min(len(sales), len(returns))

    if n > 0:
        to_remove.extend(sales.iloc[:n].index)
        to_remove.extend(returns.iloc[:n].index)

# REMOVE MATCHED SALE + RETURN
df = df.drop(index=to_remove).reset_index(drop=True)

print("Removed Sale-Return matched rows:", len(to_remove))

# ================================
# CONTACT EXTRACTION
# ================================
def extract_all_contacts(row):
    phones, emails = [], []

    try:
        ids = json.loads(row['profile.all_identities'])
    except:
        ids = []

    for i in ids:
        if "@" in str(i):
            emails.append(str(i).lower())
        else:
            phone = clean_phone(i)
            if phone:
                phones.append(phone)

    if pd.notna(row.get('profile.phone')):
        p = clean_phone(row['profile.phone'])
        if p:
            phones.append(p)

    if pd.notna(row.get('profile.identity')):
        p = clean_phone(row['profile.identity'])
        if p:
            phones.append(p)

    if pd.notna(row.get('profile.email')):
        e = extract_email(row['profile.email'])
        if e:
            emails.append(e)

    return list(set(phones)), list(set(emails))

df[['phones', 'emails']] = df.apply(lambda r: pd.Series(extract_all_contacts(r)), axis=1)

# ================================
# NAME EXTRACTION
# ================================
def extract_names(row):
    names = []

    if pd.notna(row.get('profile.name')):
        names.append(str(row['profile.name']).strip())

    if pd.notna(row.get('profile.profileData.customername')):
        names.append(str(row['profile.profileData.customername']).strip())

    return list(set([n.lower() for n in names if n]))

df['names'] = df.apply(extract_names, axis=1)

# ================================
# DYNAMIC COLUMNS
# ================================
max_phones = df['phones'].apply(len).max()
max_emails = df['emails'].apply(len).max()
max_names = df['names'].apply(len).max()

for i in range(max_phones):
    df[f'phone{i+1}'] = df['phones'].apply(lambda x: x[i] if i < len(x) else None)

for i in range(max_emails):
    df[f'email{i+1}'] = df['emails'].apply(lambda x: x[i] if i < len(x) else None)

for i in range(max_names):
    df[f'name{i+1}'] = df['names'].apply(lambda x: x[i] if i < len(x) else None)

df['primary_name'] = df['name1']

# ================================
# DATE CONVERSION
# ================================
df['invoice_datetime'] = pd.to_datetime(df['event_props.Invoice Date'], unit='s', errors='coerce')
df['final_datetime'] = pd.to_datetime(df['ts'], format='%Y%m%d%H%M%S', errors='coerce')

# ================================
# FINAL OUTPUT
# ================================
base_cols = [
    'event_props.CUID Number',
    'event_props.Billing Store Zone',
    'event_props.Invoice Number',
    'event_props.Invoice Type',
    'event_props.Store City',
    'event_props.Store Code',
    'event_props.Tender Name',
    'Invoice Brand',
    'Model Number',
    'MRP',
    'Billing Value',
    'Billing Discount Percentage',
    'primary_name',
    'invoice_datetime',
    'final_datetime'
]

phone_cols = [c for c in df.columns if c.startswith('phone')]
email_cols = [c for c in df.columns if c.startswith('email')]
name_cols = [c for c in df.columns if c.startswith('name')]

final_df = df[base_cols + name_cols + phone_cols + email_cols]

# ================================
# EXPORT
# ================================
final_df.to_csv("final_cleaned_no_returns.csv", index=False)
files.download("final_cleaned_no_returns.csv")

print("✅ Done: Sale-return adjusted dataset ready")